# 01 DEMs

ICESat-2 ATL06 download and outlier removal, coregistration of the SRTM 1999 and IfSAR 2002/2012 DEMs, and ICESat-2-derived DEMs for 2019 and 2022.

Writes `../Data_Paper/ICESat-2/icesat_2_data.shp`, `../Data_Paper/ICESat-2/icesat_2_data_no_outliers.shp`, `../Data_Paper/DEMs/d1999_coregistered.tif`, `d2002_coregistered.tif`, `d2012_coregistered.tif`, `d2019_icesat2.tif`, `d2022_icesat2.tif`, `dem_dates_icesat2.json`, and `../Data_Paper/ICESat-2/seasonal_correction/`.

In [ ]:
import logging
from datetime import datetime

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio as rio
import geoutils as gu
import xdem
from scipy.interpolate import RBFInterpolator, interp1d
from sliderule import sliderule, icesat2
from tqdm import tqdm

# Basic configuration to simply print info
logging.basicConfig(level=logging.INFO)

## 1) Download ICESat-2 ATL06 (SlideRule)

ATL06 points over the 2002 DEM footprint, keeping `atl06_quality_summary == 0`. Writes `../Data_Paper/ICESat-2/icesat_2_data.shp`.

In [ ]:
# Open example DEM
dem = xdem.DEM("../External_Data/raw_DEMs/2002-08-04.tif")

# Define region of interest as DEM footprint
bounds = list(dem.get_bounds_projected(4326))
region = sliderule.toregion(bounds)["poly"]

# Initialize SlideRule client
sliderule.init("slideruleearth.io")

# Define desired parameters for ICESat-2 ATL06
params = {
    "poly": region,
    "srt": icesat2.SRT_LAND,  # Surface-type
    "cnf": icesat2.CNF_SURFACE_HIGH,  # Confidence level
    "ats": 20.0,  # Minimum along-track spread
    "cnt": 10,  # Minimum count
}

# Request ATL06 subsetting in parallel
gdf = icesat2.atl06sp(params)

gdf = gdf[np.isfinite(gdf["h_li"])] # Keep valid data
gdf = gdf[gdf["atl06_quality_summary"]==0]  # Keep very high-confidence data
gdf = gdf.to_crs(dem.crs)
gdf.to_file("../Data_Paper/ICESat-2/icesat_2_data.shp")

## 2) Load DEMs and stable-terrain mask

2002 DEM (ellipsoid), 2012 DEM converted from NAVD88 to the ellipsoid with GEOID09. Stable terrain: off-glacier (RGI 6.0), slope < 20 degrees, 200-1500 m.

In [ ]:
dem = xdem.DEM("../External_Data/raw_DEMs/2002-08-04.tif")

d2012 = xdem.DEM('../External_Data/raw_DEMs/2012-08-30.tif').reproject(dem)
geoid_09 = gu.Raster("../External_Data/raw_DEMs/us_noaa_geoid09_ak.tif").reproject(d2012)
d2012 = d2012 + geoid_09


# Derive inlier mask as glaciers
glacier_outlines = gu.Vector('../External_Data/RGI60/01_rgi60_Alaska.shp')
inlier_mask = ~glacier_outlines.create_mask(dem)


# Remove terrain where slope > 20 degrees, where dem is below 100m (remove trees) and above 1500m (remove unstable slopes), and where 2012-2002 is too big (glaciers unacounted for)
slope = dem.slope()
slope_mask = slope.data < 20
# Combine the inlier mask and slope mask
inlier_mask.data = inlier_mask.data & slope_mask & (dem.data > 200) & (dem.data < 1500)
# Remove invalid entries from the inlier mask
inlier_mask.data[dem.data.mask==True]=False
inlier_mask.data[d2012.data.mask==True]=False

In [ ]:
gdf = gpd.read_file('../Data_Paper/ICESat-2/icesat_2_data.shp').to_crs(dem.crs)
gdf['datetime'] = pd.to_datetime(gdf['time'], format='%Y-%m-%dT%H:%M:%S.%fZ')

## 3) Remove ICESat-2 outliers

First RBF DEM from all points (multiquadric, epsilon = 100, smoothing = 500, 1 % random subsample); points whose difference from it exceeds the 95th percentile of their 50 m elevation bin are discarded. Writes `../Data_Paper/ICESat-2/icesat_2_data_no_outliers.shp`.

In [ ]:
def interpolate_icesat2(var, pts_gdf, dem_ref, factor, gridsize, function, epsilon=None, sm = 0):

    """
    Interpolate DataFrame data onto a regular grid using RBF interpolation.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        DataFrame with columns 'X', 'Y', 'h_li'
    gridsize : int
        Size of the output grid (gridsize x gridsize)
    function : str
        RBF function type ('thin_plate_spline', 'multiquadric', 'inverse_multiquadric', 
                          'inverse_quadratic', 'gaussian', 'linear', 'cubic', 'quintic')
    
    Returns:
    --------
    X_grid, Y_grid : numpy.ndarray
        Grid coordinates
    Z_grid : numpy.ndarray
        Interpolated values on the grid
    """

    pts_gdf['X'] = [pt.x for pt in pts_gdf.geometry]
    pts_gdf['Y'] = [pt.y for pt in pts_gdf.geometry]


    N = len(pts_gdf)
    n_keep = N // factor

    random_indices = np.random.default_rng(42).choice(N, size=n_keep, replace=False)  # fixed seed

    
    # Extract coordinates and values
    points = pts_gdf[['X', 'Y']].values[random_indices]
    values = pts_gdf[var].values[random_indices]
    # Ensure there are no NaNs
    mask = ~np.isnan(values)
    points = points[mask]
    values = values[mask]
    
    # Create regular grid
    x_min, x_max = pts_gdf['X'].min(), pts_gdf['X'].max()
    y_min, y_max = pts_gdf['Y'].min(), pts_gdf['Y'].max()
    
    x_grid = np.linspace(x_min, x_max, gridsize)
    y_grid = np.linspace(y_min, y_max, gridsize)
    X_grid, Y_grid = np.meshgrid(x_grid, y_grid)
    
    # Flatten grid for interpolation
    grid_points = np.column_stack([X_grid.ravel(), Y_grid.ravel()])
    
    # Create RBF interpolator
    rbf = RBFInterpolator(points, values, kernel=function, epsilon=epsilon, smoothing=sm)
    
    # Interpolate onto grid
    Z_flat = rbf(grid_points)
    Z_grid = Z_flat.reshape(gridsize, gridsize)
    
    # Create transform from gridx and gridy
    transform = rio.transform.from_bounds(np.min(X_grid), np.min(Y_grid), np.max(X_grid), np.max(Y_grid), Z_grid.shape[1], Z_grid.shape[0])
    rast_rbf = xdem.DEM.from_array(data=np.flip(Z_grid, axis = 0), transform=transform, crs=pts_gdf.crs, nodata=np.nan)

    rast_rbf = rast_rbf.reproject(res=dem_ref.res, bounds=dem_ref.bounds)
    rast_rbf.set_vcrs('Ellipsoid')


    return rast_rbf

In [ ]:
rast_icesat = interpolate_icesat2('h_li', gdf, dem, 100, 1000, 'multiquadric', epsilon = 100, sm = 500)

In [ ]:
gdf['X'] = [pt.x for pt in gdf.geometry]
gdf['Y'] = [pt.y for pt in gdf.geometry]

# Interpolate the DEM at the points
vals = rast_icesat.interp_points(points=(gdf['X'], gdf['Y']))

In [ ]:
# Calculate the differences
diffs = np.abs(gdf['h_li'] - vals)

# Define bins (0–5000 every 50)
bins = np.arange(0, 5000 + 50, 50)

# Assign each point to a bin
bin_ids = np.digitize(gdf['h_li'].values, bins)

# Compute 95th percentile for each bin
bin_p95 = {}
for b in np.unique(bin_ids):
    mask = bin_ids == b
    if np.any(mask):
        bin_p95[b] = np.nanpercentile(diffs[mask], 95)

# Create threshold array matching each point
thresholds = np.array([bin_p95.get(b, np.nan) for b in bin_ids])

# Final mask
outlier_mask = diffs < thresholds

# Create geodataframe without outliers
gdf_no_outliers = gdf[outlier_mask].copy()
gdf_no_outliers.to_file('../Data_Paper/ICESat-2/icesat_2_data_no_outliers.shp')

## 4) Coregister DEMs: first to ICESat-2, then to the 2002 DEM

Nuth and Kääb: SRTM 1999 and IfSAR 2002 to the filtered ICESat-2 points; IfSAR 2012 to the coregistered 2002 DEM. Second SRTM pass onto the coregistered 2002 DEM (vertical shift + Nuth and Kääb + 2nd-order deramp, stable terrain 1-1000 m). Writes `../Data_Paper/DEMs/d1999_coregistered.tif`, `d2002_coregistered.tif`, `d2012_coregistered.tif`.

In [ ]:
gdf_no_outliers = gpd.read_file('../Data_Paper/ICESat-2/icesat_2_data_no_outliers.shp')


d1999 = xdem.DEM('../External_Data/raw_DEMs/1999.tif').reproject(d2012) # Already in Ellipsoid
d1999.set_vcrs('Ellipsoid')

d2002 = dem.copy()
d2002.set_vcrs('Ellipsoid')
d2012.set_vcrs('Ellipsoid')


# Initialize coregistration pipeline
coreg_pipeline = xdem.coreg.NuthKaab()

# Fit and apply the coregistration
d1999_coregistered = coreg_pipeline.fit_and_apply(reference_elev=gdf_no_outliers, to_be_aligned_elev=d1999, inlier_mask=inlier_mask, z_name="h_li", random_state=42)
d1999_coregistered.set_vcrs('Ellipsoid')
d2002_coregistered = coreg_pipeline.fit_and_apply(reference_elev=gdf_no_outliers, to_be_aligned_elev=d2002, inlier_mask=inlier_mask, z_name="h_li", random_state=42)
d2002_coregistered.set_vcrs('Ellipsoid')


# 2nd coregistration with d2002
# We have to modify the inlier_mask to exclude NaNs
inlier_mask.data[d2012.data.mask] = False
d2012_coregistered = coreg_pipeline.fit_and_apply(reference_elev=d2002_coregistered, to_be_aligned_elev=d2012, inlier_mask=inlier_mask, random_state=42) # Coregister to d2002 because somehow icesat fails
d2012_coregistered.set_nodata(np.nan)
d2012_coregistered.set_vcrs('Ellipsoid')

In [ ]:
# 2nd Iteration of coregistration because d1999 has massive bias at low elevations

# Remove terrain where slope > 20 degrees, where dem is below 100m (remove trees) and above 1500m (remove unstable slopes), and where 2012-2002 is too big (glaciers unacounted for)
inlier_mask_2nd_iteration = ~glacier_outlines.create_mask(dem)
# Combine the inlier mask and slope mask
inlier_mask_2nd_iteration.data = inlier_mask_2nd_iteration.data & slope_mask & (dem.data < 1000) & (dem.data > 1)
# Remove invalid entries from the inlier mask
inlier_mask_2nd_iteration.data[d2012.data.mask==True]=False

pipeline = xdem.coreg.VerticalShift() + xdem.coreg.NuthKaab() + xdem.coreg.Deramp(poly_order=2)
d1999_coregistered = pipeline.fit_and_apply(reference_elev=d2002_coregistered, to_be_aligned_elev=d1999_coregistered, inlier_mask=inlier_mask_2nd_iteration, random_state=42)  # Coregister to d2002 because somehow icesat fails

In [ ]:
# Save the coregistered DEMs
d2012_coregistered.save('../Data_Paper/DEMs/d2012_coregistered.tif')
d1999_coregistered.save('../Data_Paper/DEMs/d1999_coregistered.tif')
d2002_coregistered.save('../Data_Paper/DEMs/d2002_coregistered.tif')

## 5) ICESat-2-derived DEMs (2019, 2022)

Seasonal correction. Points acquired outside the end-of-season window (25 August to 31 October of each year) are corrected for their seasonal height anomaly, measured from same-year pairs: each off-window point is matched with the nearest window point of the same year (< 100 m), and their height difference is corrected for the elevation difference between the two locations (2012 DEM) and for the long-term trend between the two dates (rate since the 2012 DEM). Monthly medians on smooth surfaces (local relief < 1 m at both points):
- clean ice: monthly median of that year where >= 40 pairs, otherwise pooled over all years with window data (2018, 2019, 2022-2024; 2020-2021 excluded, Seward surge);
- debris- and vegetation-covered ice: clean-ice offset times a factor fitted on the months with >= 100 pairs on that surface (any relief);
- other glacier ice: own pooled monthly medians where >= 40 pairs, else clean ice; points off the glacier: no correction.

Each DEM is dated at the mean date of that year's lobe points inside the window, and points are adjusted linearly to that date from the 2012 DEM (30 August 2012). dh/dt is standardized by median and NMAD in 50 m bins of 2012 elevation (values beyond 3 standard deviations discarded), reduced to the median of all points in each 200 m cell, interpolated with a multiquadric RBF (epsilon = 500, smoothing = 100), and the standardization is reversed and added to the 2012 DEM.

Writes `../Data_Paper/DEMs/d2019_icesat2.tif`, `d2022_icesat2.tif`, `../Data_Paper/DEMs/dem_dates_icesat2.json` and the seasonal offsets and pair counts in `../Data_Paper/ICESat-2/seasonal_correction/`.

In [ ]:
import json, os
from scipy.ndimage import uniform_filter, map_coordinates
from scipy.spatial import cKDTree
from rasterio.transform import from_bounds
from rasterio.warp import reproject, Resampling

T12 = pd.Timestamp('2012-08-30')
OUT_SC = '../Data_Paper/ICESat-2/seasonal_correction/'
os.makedirs(OUT_SC, exist_ok=True)

# 2012 DEM (reference) and its local relief (standard deviation in a 90 m window)
with rio.open('../Data_Paper/DEMs/d2012_coregistered.tif') as s:
    prof, T, CRS = s.profile.copy(), s.transform, s.crs
    z12 = s.read(1, masked=True).astype('float64').filled(np.nan)
z12[z12 < 0] = np.nan
_m = np.isfinite(z12); _z = np.where(_m, z12, 0); _c = np.maximum(uniform_filter(_m.astype(float), 9), 1e-6)
_mu = uniform_filter(_z, 9) / _c
relief = np.sqrt(np.maximum(uniform_filter(_z * _z, 9) / _c - _mu * _mu, 0)); relief[~_m] = np.nan

def sample(arr, x, y):
    """Bilinear value at points (same convention as xdem interp_points)."""
    col, row = ~T * (np.asarray(x), np.asarray(y))
    return map_coordinates(arr, [row, col], order=1, mode='constant', cval=np.nan)

# ICESat-2 points with surface class: 0 clean, 1 debris-covered, 2 vegetation-covered, 3 other glacier ice, 4 off glacier
pts = gpd.read_file('../Data_Paper/ICESat-2/icesat_2_data_no_outliers.shp').to_crs(CRS)
rgi = gpd.read_file('../External_Data/RGI60/01_rgi60_Alaska.shp').to_crs(CRS)
covers = pd.concat([gpd.read_file(f'../Data_Paper/vectors/{n}.shp').assign(c=k)
                    for n, k in [('Area_cleanice', 0), ('Area_moraines', 1), ('Area_forelands', 2)]]).to_crs(CRS)
on_glacier = pts.index.isin(gpd.sjoin(pts[['geometry']], rgi[['geometry']], predicate='within').index)
_j = gpd.sjoin(pts[['geometry']], covers[['c', 'geometry']], predicate='within'); _j = _j[~_j.index.duplicated()]
pts['cov'] = _j['c'].reindex(pts.index).fillna(pd.Series(np.where(on_glacier, 3, 4), index=pts.index)).astype(int)
pts['dt'] = pd.to_datetime(pts['datetime'])
pts['X'], pts['Y'] = pts.geometry.x.values, pts.geometry.y.values
pts = pd.DataFrame(pts.drop(columns='geometry'))
pts['z12'] = sample(z12, pts.X, pts.Y)
pts['relief'] = sample(relief, pts.X, pts.Y)
pts['year'], pts['month'] = pts.dt.dt.year, pts.dt.dt.month
pts['ta'] = (pts.dt - T12).dt.days / 365.25
pts = pts[np.isfinite(pts.z12)].reset_index(drop=True)
pts['rate'] = (pts.h_li - pts.z12) / pts.ta
print(len(pts), 'points with a 2012 elevation')

In [ ]:
# Seasonal anomaly from same-year pairs (off-window point vs nearest window point, < 100 m, same surface class)
def in_window(df):
    y = df.year.astype(str)
    return ((df.dt >= pd.to_datetime(y + '-08-25')) & (df.dt < pd.to_datetime(y + '-10-31') + pd.Timedelta(days=1))).values

rows = []
for y in sorted(set(pts.year) - {2020, 2021}):               # 2020-2021: Seward surge
    d = pts[(pts.year == y) & (pts['cov'] < 4)]
    w = in_window(d); ref, oth = d[w], d[~w]
    if len(ref) < 100: continue
    dist, idx = cKDTree(np.c_[ref.X, ref.Y]).query(np.c_[oth.X, oth.Y], distance_upper_bound=100)
    ok = np.isfinite(dist); o, r = oth[ok], ref.iloc[idx[ok]]
    dt_pair = (o.dt.values - r.dt.values) / np.timedelta64(1, 'D') / 365.25
    anomaly = (o.h_li.values - r.h_li.values) - (o.z12.values - r.z12.values) - r.rate.values * dt_pair
    rows.append(pd.DataFrame(dict(year=y, month=o.month.values, cov=o['cov'].values, same=o['cov'].values == r['cov'].values,
                                  smooth=(o.relief.values < 1) & (r.relief.values < 1), anomaly=anomaly)))
pairs = pd.concat(rows); pairs = pairs[pairs.same]

def monthly(sub, nmin):
    t = sub.groupby('month').anomaly.agg(['size', 'median'])
    return t[t['size'] >= nmin]['median']

def fill(s):
    """Window months (Sep, Oct) = 0; months without enough pairs interpolated (circular)."""
    s = s.copy(); s[9] = 0.0; s[10] = 0.0
    return pd.concat([s, s, s]).reset_index(drop=True).interpolate()[12:24].values

months = range(1, 13)
s = pd.Series(np.nan, index=months); m = monthly(pairs[(pairs['cov'] == 0) & pairs.smooth], 40); s.loc[m.index] = m
pooled = np.zeros(13); pooled[1:] = fill(s)                                     # clean ice, pooled years
allrel = pairs.groupby(['cov', 'month']).anomaly.agg(['size', 'median']).reset_index()
ref_clean = allrel[allrel['cov'] == 0].set_index('month')['median']
k = {}
for c in [1, 2]:                                                                 # debris-, vegetation-covered: k x clean
    sub = allrel[(allrel['cov'] == c) & (allrel['size'] >= 100)].set_index('month')['median']
    x = ref_clean.reindex(sub.index).values; ok = np.isfinite(x)
    k[c] = float(np.clip((x[ok] * sub.values[ok]).sum() / (x[ok] ** 2).sum(), 0, 1)) if ok.any() else 0.0
s = pd.Series(np.nan, index=months); m = monthly(pairs[(pairs['cov'] == 3) & pairs.smooth], 40); s.loc[m.index] = m
other = np.zeros(13); other[1:] = fill(s.fillna(pd.Series(pooled[1:], index=months)))

offsets, dem_dates, source = {}, {}, {}
for Y in [2019, 2022]:
    m = monthly(pairs[(pairs.year == Y) & (pairs['cov'] == 0) & pairs.smooth], 40)
    clean = pooled.copy(); src = {}
    for mo in months:
        if mo in (9, 10): src[mo] = 'window'
        elif mo in m.index: clean[mo] = m[mo]; src[mo] = str(Y)
        else: src[mo] = 'pooled'
    M = np.zeros((5, 13)); M[0] = clean; M[1] = k[1] * clean; M[2] = k[2] * clean; M[3] = other
    offsets[Y], source[Y] = M, src
    d = pts[(pts.year == Y) & (pts['cov'] < 3)]
    dem_dates[Y] = d[in_window(d)].dt.mean().normalize()
    print(f'{Y}: DEM date {dem_dates[Y].date()}; clean-ice offsets (m)', np.round(clean[1:], 2))
print(f'k debris = {k[1]:.2f}, k vegetation = {k[2]:.2f}')

json.dump({str(Y): str(dem_dates[Y].date()) for Y in dem_dates}, open('../Data_Paper/DEMs/dem_dates_icesat2.json', 'w'), indent=1)
pd.DataFrame({f'{Y} {n}': offsets[Y][i, 1:] for Y in offsets for i, n in enumerate(['clean', 'debris', 'vegetation', 'other glacier'])},
             index=pd.Index(months, name='month')).round(3).to_csv(OUT_SC + 'seasonal_offsets_m.csv')
pd.DataFrame(source).rename_axis('month').to_csv(OUT_SC + 'seasonal_offsets_source.csv')
pairs.groupby(['year', 'cov', 'smooth', 'month']).anomaly.agg(['size', 'median']).round(3).to_csv(OUT_SC + 'pair_counts.csv')

In [ ]:
def calculate_nmad(values):
    """NMAD = 1.4826 * median(|values - median(values)|)"""
    return 1.4826 * np.median(np.abs(values - np.median(values)))

def normalize(dhdt, zref):
    """Standardize dh/dt by median and NMAD of 50 m bins of 2012 elevation (linear interpolation between bins)."""
    bins = np.arange(0, 5000, 50); centers = (bins[:-1] + bins[1:]) / 2
    bi = np.digitize(zref, bins) - 1
    c, nm, md = [], [], []
    for i in range(len(bins) - 1):
        v = dhdt[(bi == i) & np.isfinite(dhdt)]
        if len(v) > 1: c.append(centers[i]); nm.append(calculate_nmad(v)); md.append(np.median(v))
    f_nm = interp1d(c, nm, bounds_error=False, fill_value='extrapolate')
    f_md = interp1d(c, md, bounds_error=False, fill_value='extrapolate')
    return (dhdt - f_md(zref)) / f_nm(zref), f_nm, f_md

def icesat2_dem(Y, cell=200.0, gridsize=int(8000 / 3)):
    d = pts[pts.year == Y]
    t_i = (dem_dates[Y] - T12).days / 365.25
    corr = np.where(in_window(d) | (d['cov'].values == 4), 0.0, offsets[Y][d['cov'].values, d.month.values])
    h = d.h_li.values - corr
    h_adj = d.z12.values + (h - d.z12.values) * t_i / d.ta.values          # linear change since 2012, at the DEM date
    dhdt = (h_adj - d.z12.values) / t_i
    std, f_nm, f_md = normalize(dhdt, d.z12.values)
    keep = np.isfinite(std) & (np.abs(std - np.nanmean(std)) <= 3 * np.nanstd(std))
    # median of all points in each 200 m cell
    c = pd.DataFrame(dict(i=np.floor(d.X.values[keep] / cell).astype(int), j=np.floor(d.Y.values[keep] / cell).astype(int),
                          x=d.X.values[keep], y=d.Y.values[keep], v=std[keep]))
    c = c.groupby(['i', 'j']).agg(x=('x', 'mean'), y=('y', 'mean'), v=('v', 'median')).reset_index()
    print(f'{Y}: {int(keep.sum())} points -> {len(c)} cells of {cell:.0f} m')
    rbf = RBFInterpolator(c[['x', 'y']].values, c.v.values, kernel='multiquadric', epsilon=500, smoothing=100)
    xg = np.linspace(d.X.min(), d.X.max(), gridsize); yg = np.linspace(d.Y.min(), d.Y.max(), gridsize)
    XX, YY = np.meshgrid(xg, yg)
    Z = np.full(XX.size, np.nan); flat = np.c_[XX.ravel(), YY.ravel()]
    for i in range(0, len(flat), 200000): Z[i:i + 200000] = rbf(flat[i:i + 200000])
    Z = np.flip(Z.reshape(gridsize, gridsize), axis=0)
    std10 = np.full(z12.shape, np.nan)
    reproject(Z, std10, src_transform=from_bounds(xg.min(), yg.min(), xg.max(), yg.max(), gridsize, gridsize), src_crs=CRS,
              dst_transform=T, dst_crs=CRS, resampling=Resampling.bilinear, src_nodata=np.nan, dst_nodata=np.nan)
    return (z12 + t_i * (std10 * f_nm(z12) + f_md(z12))).astype('float32')

In [ ]:
prof.update(dtype='float32', nodata=np.nan, compress='deflate')
for Y in [2019, 2022]:
    with rio.open(f'../Data_Paper/DEMs/d{Y}_icesat2.tif', 'w', **prof) as dst:
        dst.write(icesat2_dem(Y), 1)